Run this given csv from HeliScraper.py

In [ ]:
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from sklearn.model_selection import train_test_split

CSV_FILE = "helicopter_drone_urls.csv"
DATA_DIR = "drone_data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")

os.makedirs(IMAGE_DIR, exist_ok=True)

df = pd.read_csv(CSV_FILE)

drone_keywords = ["drone"]

df = df[df["search_term"].str.lower().apply(
    lambda x: any(k in x for k in drone_keywords)
)].reset_index(drop=True)

df["label"] = "drone"

image_records = []

for i, row in df.iterrows():
    image_id = f"{i + 1:07d}"
    filename = f"{image_id}.jpg"
    save_path = os.path.join(IMAGE_DIR, filename)

    try:
        url = row["image_url"]
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")
        img.save(save_path, "JPEG")

        image_records.append({
            "id": image_id,
            "label": row["label"]
        })

        print(f"Saved {filename}")

    except Exception as e:
        print(f"Failed: {row['image_url']}")
        print(e)

records = pd.DataFrame(image_records)

trainval, test = train_test_split(
    records,
    test_size=0.2,
    random_state=42,
    stratify=records["label"] if records["label"].nunique() > 1 else None
)

with open(os.path.join(DATA_DIR, "images.txt"), "w") as f:
    for _, row in records.iterrows():
        f.write(f"{row['id']} images/{row['id']}.jpg\n")

with open(os.path.join(DATA_DIR, "images_variant_trainval.txt"), "w") as f:
    for _, row in trainval.iterrows():
        f.write(f"{row['id']} {row['label']}\n")

with open(os.path.join(DATA_DIR, "images_variant_test.txt"), "w") as f:
    for _, row in test.iterrows():
        f.write(f"{row['id']} {row['label']}\n")

with open(os.path.join(DATA_DIR, "variants.txt"), "w") as f:
    for label in sorted(records["label"].unique()):
        f.write(f"{label}\n")

print("Done")

After deleting images in the data folder, run below to clean the metadata

In [2]:
import os

DATA_DIR = "helicopter_data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")

metadata_files = [
    "images.txt",
    "images_variant_trainval.txt",
    "images_variant_test.txt"
]

existing = set(os.listdir(IMAGE_DIR))

for metadata in metadata_files:
    path = os.path.join(DATA_DIR, metadata)

    cleaned_lines = []

    with open(path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 2:
            continue

        image_path = parts[1]
        filename = os.path.basename(image_path)

        if filename in existing:
            cleaned_lines.append(line)

    with open(path, "w") as f:
        f.writelines(cleaned_lines)

    
print("Done")

Done
